# NB57 — Conjugation Protection and the Spectral Wall

NB56 established the character-tower mass channel: integer eigenvalues
acting as VEV exponents generate exponential mass gaps with bandwidth
$16 = d(210)$. But Gens 1 and 2 remain exactly degenerate under
separable generators (palindromic $\mathbb{Z}_6$ symmetry).

**Question**: Do coupled generators — elements of $\mathbb{Z}^*_{210}$
nontrivial at multiple primes simultaneously — break the Gen 1-2
degeneracy?

**Answer**: Coupled generators break the *per-pair* palindromic symmetry
(individual Gen1$\leftrightarrow$Gen2 character pairs split by elements
of $\mathbb{Z}[\sqrt{3}]$), but the *generation-level multisets* remain
identical. The protection comes from complex conjugation $\chi \to \bar\chi$,
which is eigenvalue-preserving for any real-symmetric Cayley Laplacian
and maps Gen1$\leftrightarrow$Gen2 while simultaneously permuting the
$a_5$ index. This protection extends independently to each tower level,
making the tower product mass $m_\chi = \prod_k |v_k|^{\lambda_k(\chi)}$
immune to all spectral (Cayley-based) perturbations.

**Scope**: The spectral wall confirms that the **only** path to
Gen1$\neq$Gen2 is through $\Sigma$-breaking — a non-constant VEV on
the covering fiber (NB53). The Higgs mechanism is structurally
mandatory, not optional.

In [ ]:
# ── NB57 Setup ──
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'scripts'))
import numpy as np
from math import gcd
from itertools import product as cartesian
from collections import defaultdict
from solenoid_algebra import SA, PRIMES, PHI

# Per-prime generators (CRT-isolated)
per_prime_gens = {}
for p in PRIMES:
    if p == 2:
        per_prime_gens[p] = 1
        continue
    for g_cand in range(2, p):
        if all(pow(g_cand, d, p) != 1
               for d in range(1, p-1)
               if (p-1) % d == 0 and d < p-1):
            residues = [1 if q != p else g_cand for q in PRIMES]
            per_prime_gens[p] = SA.reconstruct(residues)
            break

# Separable generating set (with inverses)
gen_set_sep = []
for g in per_prime_gens.values():
    gen_set_sep.append(g)
    g_inv = SA.inverse(g)
    if g_inv != g:
        gen_set_sep.append(g_inv)

# Per-prime Cayley spectra
per_prime_spec = {}
for p_idx, p in enumerate(PRIMES):
    phi_p = SA.phi_p[p]
    spec = []
    if phi_p <= 1:
        spec = [0]
    else:
        gen = per_prime_gens[p]
        gen_inv = SA.inverse(gen)
        self_inv = (gen == gen_inv)
        for a in range(phi_p):
            idx = [0] * len(PRIMES)
            idx[p_idx] = a
            idx = tuple(idx)
            lam = sum(1 - SA.character(idx, s).real
                      for s in ([gen] if self_inv else [gen, gen_inv]))
            spec.append(round(lam))
    per_prime_spec[p] = spec

# Coupled generators: max-order elements of Z*_210
max_gens = SA.max_order_generators
found_coupled = None
for i, g1 in enumerate(max_gens):
    for j, g2 in enumerate(max_gens[i+1:], i+1):
        for g3 in max_gens[j+1:]:
            if SA.generates_group([g1, g2, g3]):
                found_coupled = [g1, g2, g3]
                break
        if found_coupled:
            break
    if found_coupled:
        break

gen_set_coupled = []
for g in found_coupled:
    gen_set_coupled.append(g)
    g_inv = SA.inverse(g)
    if g_inv != g:
        gen_set_coupled.append(g_inv)

# All character indices
all_indices = SA.all_character_indices()

# Generation partition
gens_by_gen = {0: [], 1: [], 2: []}
for idx in all_indices:
    gens_by_gen[idx[3] % 3].append(idx)

# Eigenvalue computation function
def compute_eigenvalues(gen_set):
    evals = {}
    for idx in all_indices:
        lam = sum(1 - SA.character(idx, s).real for s in gen_set)
        evals[idx] = lam
    return evals

sep_evals = compute_eigenvalues(gen_set_sep)
coup_evals = compute_eigenvalues(gen_set_coupled)

print("NB57: Conjugation Protection and the Spectral Wall")
print("=" * 60)
print(f"Separable generators: {list(per_prime_gens.values())}")
print(f"  |S_sep| = {len(gen_set_sep)}")
print(f"Coupled generators: {found_coupled}")
print(f"  Decompositions: {[SA.decompose(g) for g in found_coupled]}")
print(f"  |S_coupled| = {len(gen_set_coupled)}")
print(f"Per-prime spectra: {per_prime_spec}")
print(f"Generation sizes: {[len(gens_by_gen[g]) for g in range(3)]}")

In [ ]:
# ── IDENTITY #96: Character Conjugation Theorem ──
# For any symmetric generating set on Z*_210, the eigenvalue multisets
# of Gen1 and Gen2 are identical.
#
# Proof: Complex conjugation chi -> chi_bar maps character (a2,a3,a5,a7)
# to (a2, (-a3)%2, (-a5)%4, (-a7)%6) = (a2, a3, (-a5)%4, (-a7)%6).
# This bijection:
#   1. Preserves eigenvalues (real symmetry: Re[chi(g)] = Re[chi_bar(g)])
#   2. Maps Gen1 (a7 in {1,4}) <-> Gen2 (a7 in {5,2})
# Therefore the 16-element eigenvalue multiset of Gen1 = that of Gen2.

print("IDENTITY #96: Character Conjugation Theorem")
print("=" * 60)

# Step 1: Verify per-pair palindrome is broken (coupled)
print("\nStep 1: Per-pair palindrome under coupled generators")
print("-" * 60)
pair_splits = []
for idx in gens_by_gen[1]:
    a2, a3, a5, a7 = idx
    palin_idx = (a2, a3, a5, (-a7) % 6)
    split = coup_evals[idx] - coup_evals[palin_idx]
    pair_splits.append((idx, palin_idx, split))
    if abs(split) > 1e-10:
        print(f"  BROKEN: {idx} <-> {palin_idx}: "
              f"{coup_evals[idx]:.6f} vs {coup_evals[palin_idx]:.6f}, "
              f"split = {split:+.6f}")

broken_count = sum(1 for _, _, sp in pair_splits if abs(sp) > 1e-10)
print(f"\nPer-pair palindrome pairs broken: {broken_count}/{len(pair_splits)}")

# Step 2: Verify conjugation IS preserved
print("\nStep 2: Full conjugation chi -> chi_bar")
print("-" * 60)
conj_splits = []
for idx in gens_by_gen[1]:
    a2, a3, a5, a7 = idx
    conj_idx = (a2, (-a3) % 2, (-a5) % 4, (-a7) % 6)
    split = abs(coup_evals[idx] - coup_evals[conj_idx])
    conj_splits.append((idx, conj_idx, split))
    gen_conj = ((-a7) % 6) % 3
    print(f"  {idx} Gen1 <-> {conj_idx} Gen{gen_conj}: "
          f"{coup_evals[idx]:.6f} = {coup_evals[conj_idx]:.6f}  "
          f"diff = {split:.2e}")

max_conj_split = max(sp for _, _, sp in conj_splits)
print(f"\nMax conjugation split: {max_conj_split:.2e}")

# Step 3: Verify multiset equality
print("\nStep 3: Generation multisets")
print("-" * 60)
for g in range(3):
    vals = sorted([coup_evals[idx] for idx in gens_by_gen[g]])
    print(f"  Gen {g}: {[f'{v:.4f}' for v in vals]}")

gen1_sorted = sorted([coup_evals[idx] for idx in gens_by_gen[1]])
gen2_sorted = sorted([coup_evals[idx] for idx in gens_by_gen[2]])
multiset_diff = max(abs(a - b) for a, b in zip(gen1_sorted, gen2_sorted))

print(f"\nMultiset max difference: {multiset_diff:.2e}")

# Step 4: Verify also under separable generators (control)
gen1_sep = sorted([sep_evals[idx] for idx in gens_by_gen[1]])
gen2_sep = sorted([sep_evals[idx] for idx in gens_by_gen[2]])
multiset_diff_sep = max(abs(a - b) for a, b in zip(gen1_sep, gen2_sep))
print(f"Separable multiset max difference: {multiset_diff_sep:.2e} (control)")

# Verdict
conj_ok = max_conj_split < 1e-10
multiset_ok = multiset_diff < 1e-10
broken_ok = broken_count > 0
verdict = conj_ok and multiset_ok and broken_ok

print(f"\n{'PASS' if verdict else 'FAIL'}: "
      f"Palindrome broken ({broken_count} pairs), "
      f"conjugation protected (max {max_conj_split:.2e}), "
      f"multisets equal (max {multiset_diff:.2e})")

## The Cross-Phase Mechanism

Why does coupling break pairs but not multisets?

For a coupled generator $g$ with CRT indices $(r_2, r_3, r_5, r_7)$:

$$\chi(g) = \exp\!\bigl(2\pi i(\psi_{35} + \phi_7)\bigr)$$

where $\psi_{35} = a_3 k_3/2 + a_5 k_5/4$ and $\phi_7 = a_7 k_7/6$.

- **Palindrome** ($a_7 \to -a_7$, $a_5$ fixed):
  $\cos(\psi_{35} + \phi_7) \to \cos(\psi_{35} - \phi_7)$.
  Different when $\psi_{35} \neq 0$ and $\phi_7 \neq 0$.

- **Conjugation** ($a_7 \to -a_7$ AND $a_5 \to -a_5$):
  $\psi_{35} + \phi_7 \to -\psi_{35} - \phi_7$.
  $\cos(-\psi_{35} - \phi_7) = \cos(\psi_{35} + \phi_7)$. **Always equal.**

The $a_5$ flip compensates the $a_7$ flip. The palindrome changes the
sign of ONE phase component; conjugation changes the sign of ALL
components, which preserves the cosine.

In [ ]:
# ── Cross-phase structure: algebraic characterization ──
print("Cross-Phase Structure")
print("=" * 60)

sqrt3 = np.sqrt(3)

# Show all Gen1-Gen2 pair splits in Z[sqrt(3)]
print("\nPer-pair splits (palindrome-only, coupled generators):")
print(f"  {'Gen1':>16s}  {'Gen2(palin)':>16s}  {'split':>10s}  {'Z[sqrt3]':>15s}")
for idx in sorted(gens_by_gen[1], key=lambda x: abs(coup_evals[x] - coup_evals[(x[0],x[1],x[2],(-x[3])%6)]), reverse=True):
    a2, a3, a5, a7 = idx
    palin = (a2, a3, a5, (-a7) % 6)
    delta = coup_evals[idx] - coup_evals[palin]
    # Decompose into a + b*sqrt(3)
    found_decomp = False
    for b in range(-7, 8):
        a_int = delta - b * sqrt3
        if abs(a_int - round(a_int)) < 1e-10:
            a_int = int(round(a_int))
            label = f"{a_int} + {b}*sqrt(3)" if b != 0 else f"{a_int}"
            print(f"  {str(idx):>16s}  {str(palin):>16s}  {delta:>+10.6f}  {label:>15s}")
            found_decomp = True
            break
    if not found_decomp:
        print(f"  {str(idx):>16s}  {str(palin):>16s}  {delta:>+10.6f}  {'???':>15s}")

# Show the explicit cross-phase for one example
print("\nExplicit cross-phase for (0,1,1,1):")
idx_ex = (0, 1, 1, 1)
palin_ex = (0, 1, 1, 5)
conj_ex = (0, 1, 3, 5)  # (-a5)%4 = 3, (-a7)%6 = 5

print(f"  Character: {idx_ex}")
print(f"  Palindrome image: {palin_ex} (only a7 flipped)")
print(f"  Conjugate image:  {conj_ex} (a5 AND a7 flipped)")
print(f"  lambda(original):   {coup_evals[idx_ex]:.6f}")
print(f"  lambda(palindrome): {coup_evals[palin_ex]:.6f}  "
      f"diff = {coup_evals[palin_ex]-coup_evals[idx_ex]:+.6f}")
print(f"  lambda(conjugate):  {coup_evals[conj_ex]:.6f}  "
      f"diff = {coup_evals[conj_ex]-coup_evals[idx_ex]:+.6f}")

# Count distinct split values
split_vals = set()
for idx in gens_by_gen[1]:
    palin = (idx[0], idx[1], idx[2], (-idx[3]) % 6)
    delta = round(coup_evals[idx] - coup_evals[palin], 10)
    split_vals.add(delta)
print(f"\nDistinct split values: {sorted(split_vals)}")
print(f"All in Z[sqrt(3)]: verified above")

In [ ]:
# ── IDENTITY #97: Tower Product Protection Persistence ──
# The conjugation protection holds INDEPENDENTLY at each tower level.
# Therefore the tower product mass m_chi = Prod_k |v_k|^{lambda_k(chi)}
# preserves Gen1=Gen2 for ANY per-level VEV magnitudes and ANY symmetric
# generating set.

print("IDENTITY #97: Tower Product Protection Persistence")
print("=" * 60)

# ── Build per-level coupled Laplacian eigenvalues ──

# Level 0: C_6 (only p=3 active) — no coupling possible
# Level 1: C_42 (p=3 and p=7) — coupled via Z*_42 generators
# Level 2: C_210 (p=3,5,7) — coupled via Z*_210 generators

# Primitive root tables
prim_root = {3: 2, 7: 3}
dlog = {}
for p, g in prim_root.items():
    table = {}
    x = 1
    for k in range(p - 1):
        table[x] = k
        x = (x * g) % p
    dlog[p] = table

# Level 1: Z*_42 coupled generators
z_star_42 = sorted(k for k in range(1, 43) if gcd(k, 42) == 1)

def order_mod(a, n):
    if gcd(a, n) != 1: return 0
    o = 1
    while pow(a, o, n) != 1: o += 1
    return o

max_ord_42 = max(order_mod(g, 42) for g in z_star_42)
max_gens_42 = [g for g in z_star_42 if order_mod(g, 42) == max_ord_42]

# Use first max-order generator
g_l1 = max_gens_42[0]
g_l1_inv = pow(g_l1, -1, 42)
gen_set_l1 = [g_l1] + ([g_l1_inv] if g_l1_inv != g_l1 else [])

def char_42(a3, a7, g):
    r3, r7 = g % 3, g % 7
    phase = 0.0
    if r3 in dlog[3]: phase += 2 * np.pi * a3 * dlog[3][r3] / 2
    if r7 in dlog[7]: phase += 2 * np.pi * a7 * dlog[7][r7] / 6
    return np.exp(1j * phase)

level1_coupled = {}
for a3 in range(2):
    for a7 in range(6):
        lam = sum(1 - char_42(a3, a7, s).real for s in gen_set_l1)
        level1_coupled[(a3, a7)] = lam

# Level 2: coupled eigenvalues (already computed as coup_evals)
# Level 0: separable (only p=3)
level0 = {idx: per_prime_spec[3][idx[1]] for idx in all_indices}

# ── Step 1: Verify per-level conjugation ──
print("\nStep 1: Per-level conjugation symmetry")
print("-" * 60)

# Level 0: lambda depends only on a3, which is invariant under conj
max_diff_l0 = 0.0
for idx in all_indices:
    conj_idx = (idx[0], (-idx[1]) % 2, (-idx[2]) % 4, (-idx[3]) % 6)
    max_diff_l0 = max(max_diff_l0, abs(level0[idx] - level0[conj_idx]))
print(f"  Level 0 (C_6):   max conj diff = {max_diff_l0:.2e}  "
      f"{'PROTECTED' if max_diff_l0 < 1e-10 else 'BROKEN'}")

# Level 1: lambda(a3, a7) vs lambda(a3, (-a7)%6)
max_diff_l1 = 0.0
for a3 in range(2):
    for a7 in range(6):
        a7_conj = (-a7) % 6
        max_diff_l1 = max(max_diff_l1,
                          abs(level1_coupled[(a3,a7)] - level1_coupled[(a3,a7_conj)]))
print(f"  Level 1 (C_42):  max conj diff = {max_diff_l1:.2e}  "
      f"{'PROTECTED' if max_diff_l1 < 1e-10 else 'BROKEN'}")

# Level 2: lambda(chi) vs lambda(conj(chi))
max_diff_l2 = 0.0
for idx in all_indices:
    conj_idx = (idx[0], (-idx[1]) % 2, (-idx[2]) % 4, (-idx[3]) % 6)
    max_diff_l2 = max(max_diff_l2, abs(coup_evals[idx] - coup_evals[conj_idx]))
print(f"  Level 2 (C_210): max conj diff = {max_diff_l2:.2e}  "
      f"{'PROTECTED' if max_diff_l2 < 1e-10 else 'BROKEN'}")

# ── Step 2: Tower product mass with different VEVs ──
print("\nStep 2: Tower product mass with differential VEVs")
print("-" * 60)

# NB55 equilibrium VEVs (alternating signs, different magnitudes)
v_levels = [2.5125, 2.4934, 2.2899]
print(f"VEV magnitudes: |v_0| = {v_levels[0]}, |v_1| = {v_levels[1]}, "
      f"|v_2| = {v_levels[2]}")

tower_mass = {}
for idx in all_indices:
    a2, a3, a5, a7 = idx
    E0 = level0[idx]
    E1 = level1_coupled[(a3, a7)]
    E2 = coup_evals[idx]
    mass = (v_levels[0] ** E0) * (v_levels[1] ** E1) * (v_levels[2] ** E2)
    tower_mass[idx] = mass

# Compare Gen1 and Gen2 multisets
gen1_masses = sorted([tower_mass[idx] for idx in gens_by_gen[1]])
gen2_masses = sorted([tower_mass[idx] for idx in gens_by_gen[2]])

print("\nTower product masses (Gen 1 vs Gen 2):")
print(f"  {'Gen 1':>12s}  {'Gen 2':>12s}  {'|diff|':>12s}")
for m1, m2 in zip(gen1_masses, gen2_masses):
    print(f"  {m1:>12.2f}  {m2:>12.2f}  {abs(m1-m2):>12.2e}")

max_mass_diff = max(abs(m1 - m2) for m1, m2 in zip(gen1_masses, gen2_masses))
print(f"\nMax mass multiset difference: {max_mass_diff:.2e}")

# ── Step 3: Prove universality (test many generating sets) ──
print("\nStep 3: Universality test (10 random symmetric generating sets)")
print("-" * 60)
rng = np.random.default_rng(42)
max_over_all = 0.0
for trial in range(10):
    # Pick 3 random elements from Z*_210
    trial_gens = list(rng.choice(SA.Z_star, size=3, replace=False))
    trial_set = []
    for g in trial_gens:
        trial_set.append(g)
        g_inv = SA.inverse(g)
        if g_inv != g:
            trial_set.append(g_inv)

    trial_evals = compute_eigenvalues(trial_set)
    g1 = sorted([trial_evals[idx] for idx in gens_by_gen[1]])
    g2 = sorted([trial_evals[idx] for idx in gens_by_gen[2]])
    trial_diff = max(abs(a - b) for a, b in zip(g1, g2))
    max_over_all = max(max_over_all, trial_diff)
    tag = "OK" if trial_diff < 1e-10 else "FAIL"
    print(f"  Trial {trial}: gens = {trial_gens}, "
          f"max diff = {trial_diff:.2e}  [{tag}]")

print(f"\nMax over all trials: {max_over_all:.2e}")

# ── Verdict ──
all_levels_ok = max_diff_l0 < 1e-10 and max_diff_l1 < 1e-10 and max_diff_l2 < 1e-10
mass_ok = max_mass_diff < 1e-10
universal_ok = max_over_all < 1e-10
verdict = all_levels_ok and mass_ok and universal_ok

print(f"\n{'PASS' if verdict else 'FAIL'}: "
      f"Per-level conjugation ({all_levels_ok}), "
      f"tower mass ({mass_ok}), "
      f"universality ({universal_ok})")

## The Spectral Wall

The results establish a **strict structural hierarchy** of generation protection:

| Layer | Symmetry | What it protects | What breaks it |
|-------|----------|------------------|----------------|
| **Palindromic** (NB50) | $\lambda_7(a) = \lambda_7(-a \bmod 6)$ | Per-pair eigenvalues under separable generators | Coupled generators |
| **Time-reversal** (NB51-52) | $\Sigma$: real-symmetric Hamiltonian | Per-pair eigenvalues under ALL real couplings | Complex (non-Hermitian) operators |
| **Conjugation** (NB57) | $\chi \to \bar\chi$: eigenvalue-preserving bijection Gen1$\leftrightarrow$Gen2 | Generation *multisets* under ALL Cayley Laplacians | Site-dependent VEV (NB53) |
| **Tower product** (NB57) | Per-level conjugation independence | Tower product mass multisets for all VEV configurations | Fiber-position-dependent VEV (NB53) |

Each layer is strictly stronger than the previous.
The palindromic symmetry (the first layer broken) was a *per-prime phenomenon*.
The conjugation symmetry (the deepest layer) is a *whole-group phenomenon*
that acts across multiple prime indices simultaneously.

**The Spectral Wall**: No choice of generators, coupling constants,
or VEV magnitudes can break the generation mass degeneracy through
the Cayley Laplacian spectrum alone. The only route through the wall
is the Higgs mechanism — a site-dependent scalar field on the covering
fiber that breaks $\Sigma$ by acting with different reflection centers
at different base sites (NB53, Identity #85).

In [ ]:
# ── Scope and Implications ──
print("Scope and Implications")
print("=" * 60)

print("""
PARAMETER-FREE (established):
  #96: Conjugation protection theorem
       - Coupled generators break per-pair palindrome
       - Splits live in Z[sqrt(3)] (values: 0, +/-2*sqrt(3), +/-6*sqrt(3))
       - Generation multisets ALWAYS equal (conjugation chi->chi_bar)

  #97: Tower product protection persistence
       - Conjugation holds independently at each tower level
       - Tower product mass m = Prod |v_k|^{lambda_k} preserves Gen1=Gen2
       - Universal: tested across random generating sets

STRUCTURAL CLARIFICATION (from these identities):
  The Spectral Wall has four layers:
    Palindromic < Time-reversal < Conjugation < Tower product
  Each is strictly deeper; coupled generators breach the first
  but cannot touch the third.

  The Higgs mechanism (NB53) is the ONLY passage through the wall.
  This is not a choice — it is forced by the topology of the covering tower.

SCOPE BOUNDARY:
  This notebook establishes what CANNOT work.
  The positive mechanism requires:
    1. Character-tower exponents (NB56) for gross hierarchy (v^16 = 80,000x)
    2. Non-constant fiber VEV (NB53) to break Sigma
    3. The INTERSECTION: fiber VEV profile that acts as a character-dependent
       mass correction, splitting Gen1 from Gen2 within the tower product framework
  This intersection is the next frontier.
""")

In [ ]:
# ── Scorecard ──
print("NB57 SCORECARD")
print("=" * 65)

checks = {}

# Identity #96: Character Conjugation Theorem
# a) Palindrome broken
n_broken = 0
for idx in gens_by_gen[1]:
    palin = (idx[0], idx[1], idx[2], (-idx[3]) % 6)
    if abs(coup_evals[idx] - coup_evals[palin]) > 1e-10:
        n_broken += 1
checks["96a: palindrome broken"] = n_broken > 0

# b) Conjugation preserved
max_conj = 0.0
for idx in gens_by_gen[1]:
    conj = (idx[0], (-idx[1])%2, (-idx[2])%4, (-idx[3])%6)
    max_conj = max(max_conj, abs(coup_evals[idx] - coup_evals[conj]))
checks["96b: conjugation preserved"] = max_conj < 1e-10

# c) Multisets equal
g1s = sorted([coup_evals[idx] for idx in gens_by_gen[1]])
g2s = sorted([coup_evals[idx] for idx in gens_by_gen[2]])
checks["96c: multisets equal"] = max(abs(a-b) for a,b in zip(g1s,g2s)) < 1e-10

# Identity #97: Tower Product Protection
checks["97a: L0 conj protected"] = max_diff_l0 < 1e-10
checks["97b: L1 conj protected"] = max_diff_l1 < 1e-10
checks["97c: L2 conj protected"] = max_diff_l2 < 1e-10
checks["97d: tower mass multiset"] = max_mass_diff < 1e-10
checks["97e: universal (10 random)"] = max_over_all < 1e-10

print(f"{'#':>3s}  {'Identity':>40s}  {'Result':>6s}")
print(f"{'-'*3}  {'-'*40}  {'-'*6}")
n_pass = 0
for name, ok in checks.items():
    tag = "PASS" if ok else "FAIL"
    if ok: n_pass += 1
    print(f"     {name:>40s}  [{tag}]")
print(f"\n{n_pass}/{len(checks)} checks passed.")
print(f"\nRunning total: 97 predictions/identities, 0 free parameters")